# WaterOps RAG — Pipeline Demo & Evaluation

This notebook demonstrates the full RAG pipeline:
1. Document loading and chunking
2. Vector store construction
3. Query execution with source citation
4. Retrieval quality evaluation

In [ ]:
import os, sys
sys.path.insert(0, '../src')

# Set your OpenAI key
os.environ['OPENAI_API_KEY'] = 'sk-...'  # Replace with your key

from rag_pipeline import load_documents, build_vectorstore, build_rag_chain, query

## Step 1: Load and index documents

In [ ]:
from pathlib import Path

docs = load_documents(Path('../data/docs'))
print(f'Loaded {len(docs)} documents')
for d in docs:
    print(f'  - {d.metadata.get("source", "?")} ({len(d.page_content)} chars)')

## Step 2: Build vector store (embed + index)

In [ ]:
vectorstore = build_vectorstore(docs, persist=True)
print('Vector store ready.')

## Step 3: Run example queries

In [ ]:
chain, retriever = build_rag_chain(vectorstore=vectorstore)

test_questions = [
    'What are the safe pressure thresholds for the distribution network?',
    'Describe the chlorination dosing procedure step by step.',
    'How often should turbidity meters be calibrated, and what standards are used?',
    'What should I do if the high pressure alarm triggers?',
]

for q in test_questions:
    print(f'\n{'='*60}')
    print(f'Q: {q}')
    result = query(q, chain=chain, retriever=retriever)
    print(f'A: {result["answer"]}')
    print(f'Sources: {[d.metadata.get("source", "?") for d in result["source_docs"]]}')

## Step 4: Evaluate retrieval quality

In [ ]:
# Simple hit-rate evaluation: does the right document appear in top-k results?
eval_set = [
    {'question': 'What is the emergency shutdown pressure threshold?',
     'expected_source': 'pressure_management_manual.txt'},
    {'question': 'What PPE is required when handling liquid chlorine?',
     'expected_source': 'treatment_procedures.txt'},
    {'question': 'How long is the required pump ramp-up sequence?',
     'expected_source': 'pressure_management_manual.txt'},
]

hits = 0
for item in eval_set:
    retrieved = retriever.invoke(item['question'])
    sources = [Path(d.metadata.get('source', '')).name for d in retrieved]
    hit = item['expected_source'] in sources
    hits += int(hit)
    print(f'{"HIT" if hit else "MISS"}  {item["question"]}')
    print(f'       Retrieved: {sources}')

print(f'\nHit rate: {hits}/{len(eval_set)} = {hits/len(eval_set)*100:.0f}%')